<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=339991332" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD, SESSION RESTART: Stage 0 through Stage 2 setup + Stage 7 (ResNet50 only) =====

# ---------- CELL 0a/0b: bootstrap ----------
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random, glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
print("GPU:", tf.config.list_physical_devices('GPU'))

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'
WEIGHTS_DIR   = '/kaggle/input/datasets/asivakumarnair/all-best-models/'
WORK_DIR      = '/kaggle/working/'

GRADES      = ['0','1','2','3','4']
NUM_CLASSES = 5
IMG_SIZE    = 224
BATCH_SIZE  = 32
SUBSAMPLE_SEED = 42
EYEPACS_TARGET = 3662

PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5
EARLYSTOP_PAT = 7
MONITOR       = 'val_accuracy'

AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

W_RES = 'dr_best_resnet50.keras'

# ---------- CELL 1a: LOAD ALL THREE SOURCES ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['source']     = 'aptos'
aptos['patient_id'] = None

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')
assert eyepacs['patient_id'].isna().sum() == 0, "EyePACS patient_id extraction failed"

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'
messidor['patient_id'] = None

for name, df in [('APTOS', aptos), ('EyePACS', eyepacs), ('Messidor', messidor)]:
    s = df['image_path'].sample(min(200, len(df)), random_state=SEED)
    present = s.apply(os.path.exists).sum()
    print(f"{name} path check: {present}/{len(s)} present  | rows: {len(df)}")
    assert present == len(s), f"{name} image paths do not resolve"

# ---------- CELL 1b: DISJOINTNESS GATE ----------
a_ids = set(aptos['id_code'].astype(str))
e_ids = set(eyepacs['image'].astype(str))
m_ids = set(messidor['id_code'].astype(str))
assert len(a_ids & e_ids) == 0, "APTOS/EyePACS OVERLAP"
assert len(a_ids & m_ids) == 0, "APTOS/Messidor overlap"
assert len(e_ids & m_ids) == 0, "EyePACS/Messidor overlap"
print("Source disjointness: PASS")

# ---------- CELL 1c: PAIRING VERIFICATION GATE (D-dr-9) ----------
is_im = ~messidor['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
im_check = messidor[is_im].copy()
im_check['im_num'] = im_check['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
im_check = im_check.sort_values('im_num').reset_index(drop=True)
im_check['pid'] = im_check.index // 2
sizes = im_check.groupby('pid').size()
full_pairs = im_check[im_check['pid'].isin(sizes[sizes == 2].index)]
agreement = full_pairs.groupby('pid')['grade'].apply(lambda g: g.iloc[0] == g.iloc[1])
print(f"Pairing agreement: {agreement.mean():.3f} (expect ~0.749)")
assert abs(agreement.mean() - 0.749) < 0.01, "Pairing evidence did not reproduce"

# ---------- CELL 1d: EYEPACS SUBSAMPLE + POOL ----------
def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)

# ---------- CELL 2a: SAFE SPLIT WRAPPER ----------
def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

# ---------- CELL 2b: PER-SOURCE SPLITS ----------
def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    tr, va, te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), f"{tag} PATIENT LEAKAGE"
    print(f"{tag} patient-leakage check: PASS")
    return tr, va, te

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), "MESSIDOR IM PATIENT LEAKAGE"
    print(f"{tag} IM-style patient-leakage check: PASS ({len(im_df)} images, {im_df['patient_id'].nunique()} groups)")
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")
e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")
m_tr, m_va, m_te = split_messidor_mixed(messidor)

train_df = pd.concat([a_tr, e_tr, m_tr], ignore_index=True)
val_df   = pd.concat([a_va, e_va, m_va], ignore_index=True)
test_df  = pd.concat([a_te, e_te, m_te], ignore_index=True)
print(f"Pooled Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

# ---------- CELL 2c: POOLED CLASS WEIGHTS ----------
cls = np.array(GRADES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['grade'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}
print("pooled class_weight:", {c: round(w,3) for c,w in zip(cls, cw)}, f"| span {cw.max()/cw.min():.1f}x")

# ---------- CELL 2d: GENERATOR FACTORY ----------
def make_gens(preprocess_fn, tr_df=None, va_df=None, te_df=None):
    tr_df = train_df if tr_df is None else tr_df
    va_df = val_df   if va_df is None else va_df
    te_df = test_df  if te_df is None else te_df
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

# ---------- CELL 3b: MODEL BUILDER ----------
def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(),
                         Dense(256,activation='relu'), Dropout(0.3),
                         Dense(num_classes,activation='softmax')])
    return model, base

# ---------- TWO-PHASE TRAINING, PHASE-CHECKPOINTED ----------
def train_two_phase(base_class, preprocess_fn, name, phase1_ckpt, phase1_log,
                     phase2_ckpt, phase2_log, class_weight=CLASS_WEIGHT):
    model, base = build_pretrained(base_class)
    tr, va, te  = make_gens(preprocess_fn)

    base.trainable = False
    model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== {name}: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
    model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=class_weight,
              callbacks=[ModelCheckpoint(f'/kaggle/working/{phase1_ckpt}', monitor=MONITOR, save_best_only=True),
                         CSVLogger(f'/kaggle/working/{phase1_log}', append=False)], verbose=1)
    print(f"{name} Phase 1 saved: {phase1_ckpt} / {phase1_log}")

    base.trainable = True
    model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== {name}: PHASE 2 (full fine-tune, up to 60 epochs) =====")
    model.fit(tr, validation_data=va, epochs=60, class_weight=class_weight,
              callbacks=[EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
                         ModelCheckpoint(f'/kaggle/working/{phase2_ckpt}', monitor=MONITOR, save_best_only=True),
                         CSVLogger(f'/kaggle/working/{phase2_log}', append=False)], verbose=1)
    test_result = model.evaluate(te, verbose=0)
    print(f"{name} TEST:", test_result)
    print(f"{name} Phase 2 saved: {phase2_ckpt} / {phase2_log}")
    return model, test_result

res_model, res_test = train_two_phase(
    ResNet50, res_pre, "ResNet50",
    'dr_phase1_resnet50.keras', 'dr_phase1_resnet50_log.csv', W_RES, 'dr_resnet50_log.csv')

print("\n===== SUMMARY =====")
print("ResNet50 TEST:", res_test)

Seed 42 set, TF 2.20.0, tf.keras module: tensorflow.keras
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
APTOS path check: 200/200 present  | rows: 3662
EyePACS path check: 200/200 present  | rows: 35126
Messidor path check: 200/200 present  | rows: 1744
Source disjointness: PASS
Pairing agreement: 0.749 (expect ~0.749)
EyePACS patient-leakage check: PASS
WARNING [Messidor IM second]: stratified split failed, falling back to unstratified.
  sklearn error: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
Messidor IM-style patient-leakage check: PASS (687 images, 344 groups)
Pooled Train 6,344 | Val 1,361 | Test 1,363
pooled class_weight: {np.str_('0'): np.float64(0.329), np.str_('1'): np.float64(2.063), np.str_('2'): np.float64(0.952), np.str_('3'): np.float64(5.116), np.str_('4'): np.float64(4.436)} | span 15.6x


I0000 00:00:1785791684.614387      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785791684.617582      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 [==============================] - 1s 0us/step
Found 6344 validated image filenames belonging to 5 classes.
Found 1361 validated image filenames belonging to 5 classes.
Found 1363 validated image filenames belonging to 5 classes.

===== ResNet50: PHASE 1 (head only, 10 epochs) =====
Epoch 1/10


I0000 00:00:1785791713.775013      68 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


199/199 [==============================] - 579s 3s/step - loss: 1.4854 - accuracy: 0.4565 - auc: 0.7561 - val_loss: 1.1833 - val_accuracy: 0.5136 - val_auc: 0.8176
Epoch 2/10
199/199 [==============================] - 404s 2s/step - loss: 1.2867 - accuracy: 0.5328 - auc: 0.8172 - val_loss: 1.1577 - val_accuracy: 0.5136 - val_auc: 0.8267
Epoch 3/10
199/199 [==============================] - 400s 2s/step - loss: 1.2293 - accuracy: 0.5583 - auc: 0.8385 - val_loss: 0.9926 - val_accuracy: 0.6481 - val_auc: 0.8816
Epoch 4/10
199/199 [==============================] - 396s 2s/step - loss: 1.1925 - accuracy: 0.5749 - auc: 0.8472 - val_loss: 1.1645 - val_accuracy: 0.5489 - val_auc: 0.8257
Epoch 5/10
199/199 [==============================] - 400s 2s/step - loss: 1.1830 - accuracy: 0.5700 - auc: 0.8439 - val_loss: 0.9564 - val_accuracy: 0.6458 - val_auc: 0.8836
Epoch 6/10
199/199 [==============================] - 398s 2s/step - loss: 1.1410 - accuracy: 0.5794 - auc: 0.8530 - val_loss: 1.0589 - 